# Customer Segmentation with RFM and K-Means

## Objectives

This notebook will:

- create customer-level Recency, Frequency and Monetary features;
- prepare the highly skewed RFM features for machine learning;
- evaluate different numbers of clusters;
- implement K-Means customer segmentation;
- profile and name the resulting customer segments;
- recommend targeted marketing actions for each segment;
- export customer segments for the Streamlit dashboard.

## Analytical problem

The retailer needs an evidence-based method for grouping customers according to their purchasing behaviour. The resulting segments can support more relevant marketing campaigns, retention activity, and customer-value strategies.

## Methodology selection

Customer segmentation is an unsupervised machine-learning problem because the dataset does not contain existing customer-segment labels or a known target variable.

K-Means clustering was selected because:

- it is suitable for grouping numeric customer features;
- its clusters can be profiled and explained to business users;
- it is widely supported by Scikit-learn;
- it can assign each customer to one distinct segment;
- the result can be integrated into an interactive Streamlit dashboard.

RFM features were selected because they represent three important aspects of customer behaviour:

- **Recency:** days since the customer's most recent completed purchase;
- **Frequency:** number of unique completed invoices;
- **Monetary:** total completed-sales revenue generated by the customer.

Customer identifier `15287` has already been excluded from the customer-sales dataset because it appears to represent multiple unknown customers.

In [1]:
import os
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
project_root = Path.cwd()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

customer_sales_path = (
    project_root
    / "data"
    / "processed"
    / "customer_sales.parquet"
)

assert customer_sales_path.exists()

customer_sales = pd.read_parquet(customer_sales_path)

assert len(customer_sales) == 392_672
assert customer_sales["CustomerID"].nunique() == 4_337
assert not customer_sales["CustomerID"].eq(15_287).any()

print(f"Customer-sales rows: {len(customer_sales):,}")
print(
    "Reliable customer identifiers: "
    f"{customer_sales['CustomerID'].nunique():,}"
)

Customer-sales rows: 392,672
Reliable customer identifiers: 4,337


## RFM feature engineering

The analysis date is set to one day after the final transaction date. This makes a customer who purchased on the final recorded date have a recency of one day rather than zero days.

Frequency is calculated using unique invoices rather than product-line rows. This prevents invoices containing several products from being counted as several purchases.

In [3]:
analysis_date = (
    customer_sales["InvoiceDate"].max().normalize()
    + pd.Timedelta(days=1)
)

customer_rfm = (
    customer_sales.groupby(
        "CustomerID",
        as_index=False,
    )
    .agg(
        LastPurchase=("InvoiceDate", "max"),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("LineRevenue", "sum"),
    )
)

customer_rfm["Recency"] = (
    analysis_date
    - customer_rfm["LastPurchase"].dt.normalize()
).dt.days

customer_rfm = customer_rfm[
    [
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary",
        "LastPurchase",
    ]
]

print(f"Analysis date: {analysis_date.date()}")
customer_rfm.head()

Analysis date: 2011-12-10


,CustomerID,Recency,Frequency,Monetary,LastPurchase
0,12346,326,1,"77,183.60",2011-01-18 10:01:00
1,12347,3,7,"4,310.00",2011-12-07 15:52:00
2,12348,76,4,"1,797.24",2011-09-25 13:13:00
3,12349,19,1,"1,757.55",2011-11-21 09:51:00
4,12350,311,1,334.40,2011-02-02 16:01:00


In [4]:
rfm_statistics = customer_rfm[
    [
        "Recency",
        "Frequency",
        "Monetary",
    ]
].describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

rfm_statistics

,Recency,Frequency,Monetary
count,"4,337.00","4,337.00","4,337.00"
mean,93.08,4.27,"2,049.05"
std,100.02,7.70,"8,986.23"
min,1.00,1.00,3.75
25%,18.00,1.00,306.46
50%,51.00,2.00,668.58
75%,143.00,5.00,"1,660.88"
90%,263.40,9.00,"3,640.90"
95%,312.00,13.00,"5,792.76"
99%,369.64,30.00,"19,780.71"


### RFM distribution interpretation

Frequency and Monetary contain substantial positive skew. Most customers made relatively few purchases and generated modest revenue, while a small number of customers purchased very frequently or generated exceptionally high revenue.

Recency is also unevenly distributed. Because K-Means uses distances between observations, highly skewed variables and extreme values could dominate the clustering result.

A `log1p` transformation will reduce skew while retaining every customer. Standardisation will then place the three transformed features on comparable scales.

In [5]:
rfm_chart_data = customer_rfm.melt(
    id_vars="CustomerID",
    value_vars=[
        "Recency",
        "Frequency",
        "Monetary",
    ],
    var_name="RFMFeature",
    value_name="FeatureValue",
)

fig = px.histogram(
    rfm_chart_data,
    x="FeatureValue",
    facet_col="RFMFeature",
    facet_col_wrap=3,
    nbins=50,
    title="Customer RFM Feature Distributions",
    labels={
        "FeatureValue": "Feature value",
        "RFMFeature": "RFM feature",
    },
)

fig.update_layout(
    showlegend=False,
)

fig.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.split("=")[-1]
    )
)

fig.show()

In [6]:
rfm_feature_names = [
    "Recency",
    "Frequency",
    "Monetary",
]

rfm_model_features = customer_rfm[
    rfm_feature_names
].copy()

rfm_log_features = np.log1p(
    rfm_model_features
)

feature_scaler = StandardScaler()

scaled_rfm_features = feature_scaler.fit_transform(
    rfm_log_features
)

scaled_rfm_summary = pd.DataFrame(
    scaled_rfm_features,
    columns=rfm_feature_names,
).agg(
    [
        "mean",
        "std",
        "min",
        "max",
    ]
)

scaled_rfm_summary

,Recency,Frequency,Monetary
mean,-0.00,-0.00,-0.00
std,1.00,1.00,1.00
min,-2.42,-0.96,-4.00
max,1.58,5.86,4.73


## Selecting the number of clusters

Models containing between two and eight clusters will be compared using:

- **Inertia:** the total squared distance between observations and their assigned cluster centres. Lower values indicate more compact clusters, but inertia always decreases when more clusters are added.
- **Silhouette score:** compares cohesion within clusters with separation between clusters. Values closer to one indicate better-defined clusters.

The final choice should consider both statistical performance and whether the segments are sufficiently detailed and actionable for the business.

In [7]:
cluster_evaluation_records = []

for number_of_clusters in range(2, 9):
    candidate_model = KMeans(
        n_clusters=number_of_clusters,
        random_state=42,
        n_init=20,
    )

    candidate_labels = candidate_model.fit_predict(
        scaled_rfm_features
    )

    cluster_evaluation_records.append(
        {
            "NumberOfClusters": number_of_clusters,
            "Inertia": candidate_model.inertia_,
            "SilhouetteScore": silhouette_score(
                scaled_rfm_features,
                candidate_labels,
            ),
        }
    )

cluster_evaluation = pd.DataFrame(
    cluster_evaluation_records
)

cluster_evaluation

,NumberOfClusters,Inertia,SilhouetteScore
0,2,"6,474.11",0.43
1,3,"4,856.77",0.34
2,4,"3,924.57",0.33
3,5,"3,268.21",0.32
4,6,"2,839.52",0.32
5,7,"2,532.19",0.31
6,8,"2,324.03",0.30


In [8]:
fig = px.line(
    cluster_evaluation,
    x="NumberOfClusters",
    y="Inertia",
    markers=True,
    title="K-Means Elbow Analysis",
    labels={
        "NumberOfClusters": "Number of clusters",
        "Inertia": "Model inertia",
    },
)

fig.update_traces(
    line={"width": 3},
    marker={"size": 9},
)

fig.update_xaxes(
    dtick=1,
)

fig.show()

In [9]:
fig = px.line(
    cluster_evaluation,
    x="NumberOfClusters",
    y="SilhouetteScore",
    markers=True,
    title="K-Means Silhouette Scores",
    labels={
        "NumberOfClusters": "Number of clusters",
        "SilhouetteScore": "Silhouette score",
    },
)

fig.update_traces(
    line={"width": 3},
    marker={"size": 9},
)

fig.update_xaxes(
    dtick=1,
)

fig.show()

### Cluster-count decision

The two-cluster solution produces the highest silhouette score, indicating the strongest statistical separation. However, two clusters would provide only a broad division between higher-value and lower-value customers and would offer limited detail for targeted marketing.

The elbow chart shows that improvements in inertia become progressively smaller after approximately four clusters. A four-cluster solution retains a positive silhouette score of approximately 0.333 while allowing the retailer to distinguish several commercially meaningful patterns.

Therefore, **four clusters** will be selected as a balance between statistical separation, simplicity, and business usefulness. The lower silhouette score compared with the two-cluster model will be recorded as a limitation rather than hidden.

In [10]:
assert len(customer_rfm) == 4_337
assert customer_rfm["CustomerID"].is_unique

assert customer_rfm["Recency"].ge(1).all()
assert customer_rfm["Frequency"].ge(1).all()
assert customer_rfm["Monetary"].gt(0).all()

assert scaled_rfm_features.shape == (4_337, 3)
assert np.isfinite(scaled_rfm_features).all()

assert cluster_evaluation["NumberOfClusters"].tolist() == list(
    range(2, 9)
)

assert cluster_evaluation["SilhouetteScore"].between(
    -1,
    1,
).all()

print("RFM preparation and model evaluation validated successfully.")

RFM preparation and model evaluation validated successfully.
